# Pharma Perception Analytics Assistant

## AI-Powered Oncology Research Intelligence using Retrieval-Augmented Generation (RAG)

### Author
Aarushi Srivastava

### Objective

This project demonstrates how Retrieval-Augmented Generation (RAG) can be used to analyze oncology research and healthcare publications. The solution combines semantic search, vector databases, and Large Language Models (LLMs) to generate executive-ready insights from unstructured medical literature.

### Business Problem

Healthcare and pharmaceutical organizations generate vast amounts of research data. Traditional keyword search methods often fail to uncover meaningful relationships across documents.

This project enables users to query oncology research using natural language and receive evidence-based insights generated from semantically relevant publications.

### Tools Used

This project demonstrates key capabilities relevant to modern analytics organizations:

- Semantic Search
- Retrieval-Augmented Generation (RAG)
- Knowledge Synthesis
- Thematic Analysis
- AI-driven Insight Generation
- Large Language Models (LLMs)



In [ ]:
!pip install -q sentence-transformers chromadb google-generativeai pandas

## Data Loading and Initial Exploration

The dataset consists of oncology research articles and healthcare news records containing:

- Title
- Summary
- Topics
- Source
- Publication Date

The objective of this step is to understand the dataset structure and assess data quality before building the retrieval system.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "oncology_2026.csv",
    sep=";"
)

print("Rows:", len(df))
print(df.columns.tolist())

df.head()

In [ ]:
print(df.isnull().sum())

## Data Cleaning

To improve retrieval quality:

- Records with missing summaries were removed.
- Missing topic values were replaced with "Unknown".
- Relevant analytical fields were retained for downstream processing.

The resulting dataset serves as the knowledge base for the RAG system.

In [ ]:
working_df = df[
    ["Title", "Summary", "Topics"]
].copy()

working_df = working_df.dropna(
    subset=["Summary"]
)

working_df["Topics"] = working_df["Topics"].fillna(
    "Unknown"
)

print("Rows after cleaning:", len(working_df))

working_df.head()

## Sampling Strategy

To enable rapid experimentation and reduce embedding generation time, an initial subset of 500 records was selected.

The pipeline can later be scaled to the complete dataset with minimal modifications.

In [ ]:
sample_df = working_df.sample(
    n=500,
    random_state=42
).reset_index(drop=True)

print(len(sample_df))

sample_df.head()

## Semantic Embedding Generation

Traditional keyword search struggles to identify semantically similar concepts.

To enable semantic retrieval, article summaries are transformed into dense vector representations using the Sentence Transformers model:

all-MiniLM-L6-v2

This model converts text into 384-dimensional embeddings that capture contextual meaning and similarity between documents.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

## Document Embedding Generation

To support semantic retrieval, oncology article summaries are transformed into vector embeddings.

Embeddings capture the contextual meaning of text, allowing documents discussing similar concepts to be retrieved even when different terminology is used.

For the initial prototype, embeddings are generated for a sample of 500 oncology articles and research summaries.

In [ ]:
sample_df["document_text"] = (
    sample_df["Title"].astype(str)
    + ". "
    + sample_df["Summary"].astype(str)
)

sample_df.head(2)

## Embedding Generation

To enable semantic search and retrieval, oncology articles are converted into dense vector representations using a pre-trained Sentence Transformer model.

The model captures contextual meaning beyond keyword matching, allowing retrieval of conceptually similar articles even when different terminology is used.

For this prototype, embeddings are generated using the all-MiniLM-L6-v2 model.

In [ ]:
documents = sample_df["document_text"].tolist()

print("Documents:", len(documents))
print("\nFirst document preview:\n")
print(documents[0][:500])

## Generating Semantic Embeddings

The oncology articles are transformed into dense vector embeddings using the Sentence Transformers framework.

These embeddings enable semantic similarity search, allowing retrieval based on meaning rather than exact keyword matches.

The resulting vector representations form the foundation of the Retrieval-Augmented Generation (RAG) pipeline.

In [ ]:
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print("Embedding Shape:", embeddings.shape)

## Vector Database Creation

To enable efficient semantic retrieval, document embeddings are stored in a vector database.

ChromaDB is used as the vector store for this prototype. It allows similarity-based retrieval of oncology articles using vector search rather than traditional keyword matching.

Each document is stored together with metadata including article title and topic classification.

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="oncology_articles"
)

print("Collection created successfully")

In [ ]:
collection.add(
    documents=sample_df["document_text"].tolist(),
    embeddings=embeddings.tolist(),
    ids=[f"doc_{i}" for i in range(len(sample_df))],
    metadatas=[
        {
            "title": row["Title"],
            "topic": row["Topics"]
        }
        for _, row in sample_df.iterrows()
    ]
)

print("Documents added successfully")

## Semantic Retrieval

The vector database enables retrieval based on semantic meaning rather than exact keyword matches.

A user query is converted into an embedding and compared against stored document embeddings to identify the most relevant oncology articles.

This forms the retrieval layer of the Retrieval-Augmented Generation (RAG) architecture.

In [ ]:
query = "What are emerging trends in targeted cancer therapy?"

query_embedding = embedding_model.encode(
    [query]
)

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

print("Top Results Retrieved")

In [ ]:
for i, doc in enumerate(results["documents"][0], start=1):
    print("=" * 100)
    print(f"RESULT {i}")
    print(doc[:800])
    print("\n")

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = "YOUR KEY HERE"

In [ ]:
!pip install -q google-generativeai

In [ ]:
!pip install -q -U google-genai

## LLM Integration

To generate executive-ready insights from retrieved oncology articles, Google Gemini is used as the generative component of the RAG pipeline.

The model synthesizes evidence from retrieved documents and produces concise, stakeholder-friendly summaries.

In [ ]:
from google import genai
import os

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain targeted cancer therapy in 3 bullet points."
)

print(response.text)

## Retrieval-Augmented Generation (RAG)

The RAG pipeline combines semantic retrieval with generative AI.

Instead of relying solely on an LLM's training data, the system first retrieves the most relevant oncology articles from the vector database and then uses Gemini to synthesize evidence-based insights.

This approach improves factual grounding, transparency, and relevance of generated outputs.

In [ ]:
def retrieve_documents(query, top_k=5):

    query_embedding = embedding_model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )

    return results

In [ ]:
def build_context(results):

    docs = results["documents"][0]

    context = "\n\n".join(
        [f"Document {i+1}: {doc}"
         for i, doc in enumerate(docs)]
    )

    return context

In [ ]:
def generate_insights(query):

    results = retrieve_documents(query)

    context = build_context(results)

    prompt = f"""
You are a Senior Oncology Insights Analyst.

Using ONLY the provided documents, answer the question below.

Question:
{query}

Documents:
{context}

Provide your response in the following format:

Executive Summary

Key Themes

Emerging Trends

Strategic Implications

Supporting Evidence
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [ ]:
query = """
What are emerging trends in targeted cancer therapy?
"""

answer = generate_insights(query)

print(answer)

# Conclusion

This project demonstrates an end-to-end Retrieval-Augmented Generation (RAG) pipeline for oncology research analytics.

Key capabilities include:

- Semantic search using sentence embeddings
- Vector storage and retrieval with ChromaDB
- AI-powered insight generation using Gemini
- Thematic analysis of unstructured oncology publications
- Executive-ready summaries for decision support

Future enhancements include scaling to the full dataset, integrating domain-specific biomedical embeddings, implementing source citations, and developing an interactive dashboard interface.

### DEMO

In [ ]:
query = """
What are emerging trends in targeted cancer therapy?
"""

answer = generate_insights(query)

print(answer)